#### Task 5: Key purchasing patterns (facts to give ChatGPT / Copilot for the Insights summary).

In [2]:
import os
import numpy as np
import pandas as pd
 
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
 
 
def load_data():
    """Load the real OnlineRetail.csv if it is in this folder, otherwise the simulated sample."""
    for name in ("OnlineRetail.csv", "OnlineRetail_sample.csv"):
        path = os.path.join(HERE, name)
        if os.path.exists(path):
            df = pd.read_csv(path, encoding="ISO-8859-1", dtype={"InvoiceNo": str, "StockCode": str})
            print(f"Loaded {name}: {len(df):,} rows" + ("" if name == "OnlineRetail.csv" else "  (SIMULATED STAND-IN DATA, not the real file)"))
            break
    else:
        raise FileNotFoundError("Put OnlineRetail.csv (or run generate_sample_data.py) in the same folder.")
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["LineTotal"] = df["Quantity"] * df["UnitPrice"]
    return df
 
 
def valid_rows(df):
    """Drop cancellations (InvoiceNo starts with C) and rows with non-positive quantity or price."""
    return df[~df["InvoiceNo"].str.startswith("C") & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
 
 
df = valid_rows(load_data())
total = df["LineTotal"].sum()
orders = df.groupby("InvoiceNo").agg(value=("LineTotal", "sum"), country=("Country", "first"), date=("InvoiceDate", "first"),
                                     cust=("CustomerID", "first"))
 
print("\n===== KEY FACTS =====")
print(f"Total sales: {total:,.0f} from {len(orders):,} invoices")
 
# High-value countries
by_country = df.groupby("Country")["LineTotal"].sum().sort_values(ascending=False)
print("\n1) Sales by country (top 5):")
for c, v in by_country.head(5).items():
    print(f"   {c}: {v:,.0f} ({v / total:.1%})")
 
# Average order size
aov_country = orders.groupby("country")["value"].agg(["mean", "count"]).query("count >= 10").sort_values("mean", ascending=False)
print(f"\n2) Average order value overall: {orders['value'].mean():,.2f} (median {orders['value'].median():,.2f})")
print("   Highest average order value (countries with 10+ invoices):")
for c, r in aov_country.head(3).iterrows():
    print(f"   {c}: {r['mean']:,.2f} over {int(r['count'])} invoices")
print(f"   UK average order value: {orders.loc[orders['country'] == 'United Kingdom', 'value'].mean():,.2f}")
 
# Seasonal trend
monthly = df.groupby(df["InvoiceDate"].dt.to_period("M"))["LineTotal"].sum()
peak = monthly.idxmax()
print(f"\n3) Monthly sales: peak month = {peak} ({monthly.max():,.0f}); lowest month = {monthly.idxmin()} ({monthly.min():,.0f})")
q4 = df[df["InvoiceDate"].dt.month.isin([9, 10, 11])]["LineTotal"].sum()
print(f"   Sep-Nov share of all sales: {q4 / total:.1%} (3 of 12 months = 25% if flat)")
 
# Products and big orders
top_products = df.groupby("Description")["LineTotal"].sum().sort_values(ascending=False).head(3)
print("\n4) Top 3 products by revenue: " + "; ".join(f"{d} ({v:,.0f})" for d, v in top_products.items()))
big = orders[orders["value"] > 10_000]
print(f"   Invoices above 10,000: {len(big)} ({len(big) / len(orders):.2%} of invoices) = {big['value'].sum() / total:.1%} of sales")
 
# Customer behaviour
c = orders.dropna(subset=["cust"]).groupby("cust").size()
print(f"\n5) Repeat customers: {(c > 1).mean():.1%} of identified customers placed more than one order")
print("\nPaste these facts into ChatGPT or Copilot and ask for a short summary of key purchasing patterns.")

Loaded OnlineRetail_sample.csv: 3,989 rows  (SIMULATED STAND-IN DATA, not the real file)

===== KEY FACTS =====
Total sales: 313,900 from 720 invoices

1) Sales by country (top 5):
   United Kingdom: 207,506 (66.1%)
   Germany: 47,103 (15.0%)
   EIRE: 18,307 (5.8%)
   Australia: 15,826 (5.0%)
   Netherlands: 8,421 (2.7%)

2) Average order value overall: 435.97 (median 79.81)
   Highest average order value (countries with 10+ invoices):
   Australia: 1,582.59 over 10 invoices
   Netherlands: 842.14 over 10 invoices
   Germany: 775.89 over 47 invoices
   UK average order value: 400.04

3) Monthly sales: peak month = 2011-08 (52,426); lowest month = 2011-12 (2,167)
   Sep-Nov share of all sales: 26.6% (3 of 12 months = 25% if flat)

4) Top 3 products by revenue: BULK ORDER CHRISTMAS BAUBLE (44,712); WOODEN TOY TRAIN (21,256); DOORMAT (18,621)
   Invoices above 10,000: 3 (0.42% of invoices) = 11.4% of sales

5) Repeat customers: 93.4% of identified customers placed more than one order

Pas